# 01 — Variables and Types

**Goal of this notebook:** be able to read any line in `engine/` that creates a variable or does basic arithmetic, and understand exactly what's happening.

**How to use this notebook:**
1. Read each section top to bottom.
2. Run every code cell yourself (Shift+Enter) — don't just read it. Reading lies; running tells the truth.
3. **Predict before you run.** Before pressing Shift+Enter, say out loud what you think the output will be. Get it wrong a lot. That's where learning happens.
4. The exercises at the bottom have empty cells — type your own code in them. No copy-paste.

---
## 1. What is a variable?

**Beginner terms:** A variable is a labelled box. You put a value in it, give the box a name, and from then on whenever you write that name Python looks inside the box and uses whatever's there.

**Why quant finance uses it:** Every backtest has state that changes over time — current account balance, current open position, the latest ATR reading, today's stop level. Variables are how Python remembers those things between lines of code. Without variables you couldn't write `if balance < daily_loss_limit: stop_trading()` because there'd be nothing to put in the slot called `balance`.

**The syntax:** `name = value`. The `=` is **not** equality — it's assignment. Read it as "becomes" or "is set to."

In [ ]:
# Run this cell. Then look in the next cell to see how we use these variables.
account_balance = 100000
current_position = 0
latest_atr = 35.7

In [ ]:
# In Jupyter, the LAST line of a cell is shown automatically.
# So we can just type the variable name to see what's in it:
account_balance

In [ ]:
# To show MORE than one thing in a cell, use print():
print(account_balance)
print(current_position)
print(latest_atr)

**Two important things just happened.**

1. The last line trick (typing just `account_balance`) only works in Jupyter notebooks. In a `.py` script it would do nothing. `print()` works everywhere.
2. We named our variables `account_balance`, not `accountBalance` or `AccountBalance`. The Python convention is **snake_case** — lowercase words separated by underscores. The codebase you'll be reading uses this everywhere.

---
## 2. The four basic types

Every value in Python has a **type**. The type controls what you can do with the value. Trying to subtract a string from a number is an error — but multiplying two numbers isn't.

We'll meet four basic types now. There are more, but these four account for ~90% of the variables in `engine/`.

| Type | Stands for | What it holds | Example trading uses |
|---|---|---|---|
| `int` | integer | Whole numbers, no decimals | Number of contracts, bar count, trade count |
| `float` | floating point | Decimals | Prices, ATR, percentages, P&L |
| `str` | string | Text (in quotes) | Symbol name, exit reason, side ("long"/"short") |
| `bool` | boolean | Just `True` or `False` | Is in trade? Did stop hit? Is signal valid? |

**Why types exist (the finance angle):** trade counts are integers (you can't open 2.7 contracts on NQ — it's whole contracts only). Prices are floats (15042.25 is a real NQ price). The exit reason is text ("stop_hit", "take_profit", "session_close"). Whether a trade was a winner is a yes/no — a bool. Different real-world things, different types.

In [ ]:
# int — whole numbers
contracts = 2
bars_in_trade = 17

# float — decimals
entry_price = 15042.25
atr = 35.7
win_rate = 0.54   # this is a fraction 0–1; 54% as a decimal

# str — text in quotes (single or double, doesn't matter)
symbol = "NQ"
side = 'long'
exit_reason = "stop_hit"

# bool — exactly True or False (capitalised, no quotes)
is_in_trade = True
stop_hit = False

print(contracts, entry_price, symbol, is_in_trade)

In [ ]:
# You can ask Python what type any value is, with type():
print(type(contracts))
print(type(entry_price))
print(type(symbol))
print(type(is_in_trade))

**Subtle gotcha — `int` vs `float`:**

`5` is an int. `5.0` is a float. They look the same to a human but Python treats them differently.

Run the next cell and look carefully:

In [ ]:
print(type(5))
print(type(5.0))

In modern Python (Python 3, which is what we use) this distinction matters less than it used to — most of the time the language quietly converts between them. But you'll see lines like `entry_price = 15042.25` and `contracts = 2` in the codebase, and now you know why one has a decimal and the other doesn't: they're representing fundamentally different real-world things.

---
## 3. Arithmetic

**Beginner terms:** Python uses normal-looking maths operators. The only oddities are `**` for power (not `^`) and `/` always gives you a decimal.

**Why this matters in finance:** every P&L calculation, every position-sizing rule, every drawdown number is arithmetic. If you can do these by hand you can read every numeric line in `engine/`.

| Operator | Meaning | Example |
|---|---|---|
| `+` | add | `2 + 3` → `5` |
| `-` | subtract | `10 - 4` → `6` |
| `*` | multiply | `2 * 3` → `6` |
| `/` | divide (always float result) | `10 / 4` → `2.5` |
| `//` | integer division (floor) | `10 // 4` → `2` |
| `%` | remainder | `10 % 4` → `2` |
| `**` | power | `2 ** 3` → `8` |

In [ ]:
# A small worked example: NQ futures.
# Each NQ point = $20 of P&L per contract.
# (This is a real instrument detail — full-size NQ has a $20/point multiplier.)

entry_price = 15000.00
exit_price  = 15042.50
contracts   = 2
point_value = 20  # dollars per point per contract

points_gained = exit_price - entry_price
pnl = points_gained * contracts * point_value

print("Points gained:", points_gained)
print("P&L ($):", pnl)

**What just happened, line by line:**
1. We set the entry price (a float) to 15000.00.
2. We set the exit price to 15042.50 — 42.5 points higher.
3. We're trading 2 contracts (an int — you can't trade 2.5).
4. NQ pays $20 per point per contract.
5. `points_gained = exit_price - entry_price` → 42.5
6. `pnl = 42.5 * 2 * 20` → $1700

**Why this matters:** this is a *unit P&L calculation* — and it's literally what `engine/backtester.py` does on every closed trade. When you read that file later, you'll see the same arithmetic, just buried inside a class method.

---
## 4. Comparisons (and how they make bools)

**Beginner terms:** comparisons ask Python a yes/no question. The answer comes back as a `True` or `False` — i.e. a bool.

**Why finance uses this:** every entry rule, every stop check, every halt check is a comparison. "Is the current price below the stop level?" — that's `current_price < stop_level`, and the answer is a bool you can act on.

| Operator | Meaning |
|---|---|
| `==` | equal to (NOT `=` — that's assignment!) |
| `!=` | not equal to |
| `<` | less than |
| `>` | greater than |
| `<=` | less than or equal |
| `>=` | greater than or equal |

**The single most common beginner bug:** writing `=` when you mean `==`. `=` puts a value into a box. `==` asks if two values are equal.

In [ ]:
current_price = 14985.00
stop_level    = 15000.00

stop_hit = current_price <= stop_level
print("current_price <= stop_level?", stop_hit)
print("type of stop_hit:", type(stop_hit))

In [ ]:
# Combining comparisons with `and` / `or` / `not`:
balance = 95000
max_dd_limit = 90000
consec_losses = 4
loss_limit = 5

# We can keep trading only if both: balance is above DD limit AND consec losses below limit
can_trade = (balance > max_dd_limit) and (consec_losses < loss_limit)
print("Can trade?", can_trade)

---
## 5. Reassignment (variables aren't permanent)

**Beginner terms:** once you've put a value in a variable, you can put a new value in later. The old one's gone.

**Why this matters:** the backtester walks bar-by-bar, and on each bar it updates `current_balance`, `current_position`, etc. Without reassignment there'd be no way to track state changing over time.

In [ ]:
balance = 100000
print("Start of day:", balance)

balance = balance + 500     # winning trade #1
print("After trade 1:", balance)

balance = balance - 200     # losing trade #2
print("After trade 2:", balance)

# There's a shorthand for `balance = balance + X`:  balance += X
balance += 750              # winning trade #3
print("After trade 3:", balance)

Notice line 4 — `balance = balance + 500`. **Read this as: `balance` becomes (whatever balance currently is, plus 500).** Python evaluates the right side first using the OLD value, then assigns the result back to the same name. This is one of the things that confuses people who studied algebra first, where `x = x + 500` would be nonsense.

---
## Exercises

Type the answers in the empty code cells below. **No copy-paste.** Predict the answer first, then run, then check.

If you get stuck, read the relevant section above again. If still stuck, just write `# stuck` in the cell and we'll review it together.

### Exercise 1 — P&L from a short trade

A short trade *profits* when price goes DOWN. The P&L formula is:

`pnl = (entry_price - exit_price) * contracts * point_value`

Given:
- entry price 15100.00
- exit price 15067.50
- 3 contracts
- NQ point value $20

Compute and print the P&L. (Predict the dollar amount before running.)

In [ ]:
# your code here


### Exercise 2 — risk-reward ratio

A trade has a stop 1.5 ATR below entry and a take-profit 2.0 ATR above entry. Given `atr = 35.7`:

- compute the stop distance in points
- compute the take-profit distance in points
- compute the risk:reward ratio (`tp_distance / stop_distance`)
- print all three

(Predict the ratio before running. If it surprises you, that's interesting.)

In [ ]:
# your code here


### Exercise 3 — was this a winning trade?

Given `pnl = 1700.00`, write a single line that creates a bool variable `is_winner` which is `True` if pnl > 0, else `False`. Print it. Print its `type()` to confirm it's a bool.

In [ ]:
# your code here


### Exercise 4 — risk gate

Imagine the prop firm rule: you may only place a new trade if **both** of these are true:
- account balance is at least $97,000 (you're not too close to the trailing DD)
- you haven't had 5 consecutive losses in a row

Given `balance = 96500` and `consec_losses = 3`, write a single bool variable `can_trade` and print it. (Predict first.)

In [ ]:
# your code here


### Exercise 5 — running balance (reassignment)

Start with `balance = 100000`. Apply these trades in order, using `+=` and `-=`:
1. win $850
2. lose $400
3. win $1200
4. lose $250

Print the balance after each trade. (Predict the final number first.)

In [ ]:
# your code here


---
## When you're done

Tell me you've finished and let me know:
1. Anything that surprised you, or anywhere you got something wrong on your prediction.
2. Anything that feels still hazy.
3. Whether the explanations were too long, too short, or about right.

Next notebook (`02_lists_and_loops.ipynb`) will cover lists (sequences of values — like a list of trade PnLs), `for` loops (doing something to every item in a list), and the basic `if`/`else` statement.